# Dose-Response (alpha-scaling) — Base Models

Reproduces **Figure 3** and **Table 13** (base-model dose-response): the pairing
effect Delta(S) = S(B_cult) − S(B_unrel) as the identity→item attention edges of
the binding heads are scaled by alpha ∈ {0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3}
(alpha = 1 is the unmodified model, alpha = 0 is a full knockout).

Rebuilt from `cultural-binding-heads-main/pipeline_base.ipynb` (unified base
pipeline), cells 24 (dose-response loop) and 26 (plot), plus the minimal verbatim
prerequisites from cells 6 (data loading), 8 (model loading), 10 (answer-token
discovery) and 16 (baseline S-scores). Helper functions are imported from
`common/` instead of re-defined.

Run once per `MODEL_KEY`. Outputs are written to `./results/<model>_base/`.

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma", "nemo"}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

import pickle
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

from common import config

CFG = config.init(MODEL_KEY, "base")
SEED = config.SEED
DATA_DIR = config.DATA_DIR
HF_TOKEN = config.HF_TOKEN
HEADS = config.HEADS
OUTPUT_DIR = config.OUTPUT_DIR

from common.base.data import load_n4, build_factorial_as_conditions
from common.base.prompts import (
    find_option_token_ids, format_for_base,
    discover_after_paren_ids, discover_after_paren_ids_in_prompt,
    find_identity_and_item_positions,
)
from common.base.scoring import compute_s_scores, compute_s_scores_scaled

## Dataset — N4 factorial pairs

In [ ]:
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data["B_cult"])
print(f"  {n_total} factorial pairs, {len(set(data['scenarios']))} scenarios")

## Model and tokenizer

In [ ]:
print(f"Loading {CFG['path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG["path"], trust_remote_code=True, token=HF_TOKEN
)
model = AutoModelForCausalLM.from_pretrained(
    CFG["path"], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
first_device = next(model.parameters()).device

n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
n_kv = getattr(model.config, "num_key_value_heads", n_heads)
softcap_val = getattr(model.config, "attn_logit_softcapping", None)

print(f"  {n_layers} layers, {n_heads} Q-heads, {n_kv} KV-heads (GQA group={n_heads // n_kv})")
print(f"  Softcapping: {softcap_val}")
print(f"  Search space: {n_layers * n_heads} (layer, head) pairs")


# Publish runtime singletons to common.config (required by common/ helpers)
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

## Answer-token discovery (conditional scoring)

In [ ]:
# -- Option tokens (bare scoring fallback) --
option_tokens_raw = find_option_token_ids(tokenizer)
option_tokens = {
    opt: torch.tensor(ids, device=first_device)
    for opt, ids in option_tokens_raw.items()
}

# -- Conditional scoring: discover COND_AFTER_IDS from the model --
COND_PREFIX_TID = None
COND_AFTER_IDS = None

_sample = format_for_base(data["B_cult"][:1])[0]

if CFG["cond_approach"] == "prefix":
    COND_PREFIX_TID, COND_AFTER_IDS, _sample_cov = discover_after_paren_ids(
        tokenizer, model, first_device, _sample,
        cond_token_str=CFG["cond_token_str"],
    )
    print(f"Conditional scoring: prefix {repr(CFG['cond_token_str'])} = TID {COND_PREFIX_TID}")
else:
    COND_AFTER_IDS, _sample_cov = discover_after_paren_ids_in_prompt(
        tokenizer, model, first_device, _sample,
    )
    print(f"Conditional scoring: '(' embedded in ANSWER_START (single forward pass)")

print(f"  after_ids: a={COND_AFTER_IDS['a']}, b={COND_AFTER_IDS['b']}, c={COND_AFTER_IDS['c']}")
print(f"  Sample coverage P(a+b+c | '('): {_sample_cov:.4f}")
for opt in ["a", "b", "c"]:
    decoded = [tokenizer.decode([tid]) for tid in COND_AFTER_IDS[opt]]
    print(f"  {opt}: TIDs={COND_AFTER_IDS[opt]} -> {decoded}")

## Baseline S-scores

In [ ]:
# -- Baseline scores --
conditions = ["B_cult", "B_unrel"]
texts_fmt = {c: format_for_base(data[c]) for c in conditions}

print("Computing baseline S-scores...")
scores_baseline = {}
coverages_baseline = {}
logprobs_baseline = {}
for c in conditions:
    scores_baseline[c], coverages_baseline[c], logprobs_baseline[c] = compute_s_scores(
        model, tokenizer, texts_fmt[c], COND_AFTER_IDS,
        cond_prefix_tid=COND_PREFIX_TID,
    )
    print(f"  {c}: mean S = {scores_baseline[c].mean():.4f}, "
          f"mean coverage = {coverages_baseline[c].mean():.4f}")

delta_S = scores_baseline["B_cult"].mean() - scores_baseline["B_unrel"].mean()
print(f"\n  Delta(S) = {delta_S:.4f}")
if delta_S < 0:
    print("  -> Negative: model engages cultural binding on match prompts")
else:
    print("  -> Positive or zero: weak/absent binding signal")

## Dose-response (alpha-scaling) — Figure 3 / Table 13

In [ ]:
# -- Positions --
texts_fmt_dr = {c: format_for_base(data[c]) for c in conditions}
positions_dr = {c: [] for c in conditions}
for c in conditions:
    for i in range(n_total):
        pos = find_identity_and_item_positions(
            tokenizer, texts_fmt_dr[c][i], data[c][i],
            data["items_cult"][i], data["assoc_pos"][i])
        positions_dr[c].append(pos)

# -- Run --
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
results_dr = {}

for alpha in ALPHAS:
    print(f"  alpha = {alpha:.2f} ...")
    scores = {}
    for c in conditions:
        scores[c] = compute_s_scores_scaled(
            model, tokenizer, texts_fmt_dr[c], positions_dr[c],
            HEADS, "B_to_item", COND_AFTER_IDS, alpha=alpha,
            cond_prefix_tid=COND_PREFIX_TID,
        )
    delta = scores["B_cult"].mean() - scores["B_unrel"].mean()
    results_dr[alpha] = {
        "delta": delta,
        "s_cult": scores["B_cult"].mean(),
        "s_unrel": scores["B_unrel"].mean(),
    }
    print(f"    S(cult)={results_dr[alpha]['s_cult']:.4f}, "
          f"S(unrel)={results_dr[alpha]['s_unrel']:.4f}, Delta={delta:.4f}")
    torch.cuda.empty_cache()

In [ ]:
# Added for the submission repo: persist dose-response results (Table 13 / Figure 3 data).
dr_path = OUTPUT_DIR / f"dose_response_{MODEL_KEY}_base.pkl"
with open(dr_path, "wb") as f:
    pickle.dump({"model": MODEL_KEY, "variant": "base", "heads": HEADS,
                 "alphas": ALPHAS, "results_dr": results_dr}, f)
print(f"Saved: {dr_path}")

### Dose-response plot (Figure 3)

In [ ]:
import matplotlib.pyplot as plt

alphas_list = sorted(results_dr.keys())
deltas = [results_dr[a]["delta"] for a in alphas_list]
s_cults = [results_dr[a]["s_cult"] for a in alphas_list]
s_unrels = [results_dr[a]["s_unrel"] for a in alphas_list]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(alphas_list, deltas, "o-", color="#d62728", linewidth=2, markersize=8)
ax.axhline(y=results_dr[1.0]["delta"], color="gray", ls="--", alpha=0.5, label="baseline (alpha=1)")
ax.axvline(x=1.0, color="gray", ls=":", alpha=0.3)
ax.set_xlabel("alpha (attention scale factor)")
ax.set_ylabel("Delta(S) = S(B_cult) - S(B_unrel)")
ax.set_title(f"{MODEL_KEY.upper()} BASE: pairing effect vs attention scaling")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(alphas_list, s_cults, "s-", color="#1f77b4", linewidth=2, markersize=8, label="S(B_cult)")
ax.plot(alphas_list, s_unrels, "o-", color="#ff7f0e", linewidth=2, markersize=8, label="S(B_unrel)")
ax.axvline(x=1.0, color="gray", ls=":", alpha=0.3)
ax.set_xlabel("alpha (attention scale factor)")
ax.set_ylabel("S-score")
ax.set_title(f"{MODEL_KEY.upper()} BASE: S-scores vs attention scaling")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / f"dose_response_{MODEL_KEY}_base.png",
            dpi=200, bbox_inches="tight")  # added: persist figure
plt.show()